# Lesson 4: Walking downhill. Gradient descent (Cauchy 1847) and why it works (Taylor 1715)

## The old ideas

- **Taylor series (Brook Taylor, 1715):** if you zoom in on any smooth curve far enough, it looks like a straight line. Zoom in a *little* less and it looks like a parabola. This lets you replace complicated functions by simple approximations near a point, and tells you exactly how wrong the approximation is.
- **Steepest descent (Augustin-Louis Cauchy, 1847):** to find the lowest point of a landscape when you can't solve for it directly, repeatedly take a small step in the direction that goes downhill fastest, which is the direction opposite the gradient (Lesson 2).
- **Newton's method (Newton, 1680s; Raphson, 1690):** use the curvature too, and you can sometimes jump straight to the bottom.

## What they become in machine learning

**Gradient descent is how nearly every model is trained**, from linear regression to the largest language models. "Learning rate" is Cauchy's step size. "Learning rate too high and training explodes" is a statement about curvature that Taylor's series explains. "Momentum" and "Adam" are refinements of the same walk.

**By the end of this lesson you can:** explain, with algebra, *why* stepping against the slope reduces the error and *exactly how big* a step is safe; write a general gradient-descent routine; train a line by it and verify it matches Lesson 3's exact answer; and explain why some landscapes are slow.

---
## Step 0: Setup

*Setup cell: Lesson 1's slope tool. Run as-is.*

In [ ]:
def slope_by_wiggling(function, point, gap=0.0001):
    return (function(point + gap) - function(point)) / gap

---
## Step 1: Zooming in (Taylor's idea)

Take `f(x) = x³` near `x = 2`. There `f(2) = 8`, slope `f'(2) = 3x² = 12`, and curvature `f''(2) = 6x = 12`. Taylor's claim: for a small step `h`,

```
f(2 + h)  ≈  f(2) + f'(2)·h                     (straight-line approximation)
f(2 + h)  ≈  f(2) + f'(2)·h + ½·f''(2)·h²       (parabola approximation)
```

In general: `f(x + h) ≈ f(x) + f'(x)·h + ½ f''(x)·h² + ...`. Test it. Type:

```python
def cube(x):
    return x ** 3

for step in [1, 0.5, 0.1, 0.01]:
    actual = cube(2 + step)
    line_guess = 8 + 12 * step
    parabola_guess = 8 + 12 * step + 0.5 * 12 * step ** 2
    print(step, actual, abs(actual - line_guess), abs(actual - parabola_guess))
```

In [ ]:
# type it here

**Expected output:**
```text
1 27 7 1.0
0.5 15.625 1.625 0.125
0.1 9.261000000000001 0.06100000000000172 0.0010000000000012221
0.01 8.120600999999997 0.0006009999999978533 9.999999974752427e-07
```

Columns: step size, true value, error of the line guess, error of the parabola guess. Watch the errors as the step shrinks by 10×: the line's error drops by about **100×** (`0.061 → 0.0006`), because the first thing it ignores is the `h²` term. The parabola's error drops by about **1000×** (`0.001 → 0.000001`), because the first thing *it* ignores is an `h³` term. The smaller the step, the better either approximation, and the parabola is much better.

**Why it matters for learning:** a loss function can be hideously complicated. But *near where you are*, it behaves like a straight line (slope) with a parabola correction (curvature). Everything below is reasoning about that simple local picture.

---
## Step 2: Why a downhill step works, and how big is safe

Stand at `x` with slope `g = f'(x)` and curvature `c = f''(x)`. Take a step of `h = −lr · g` (opposite the slope, scaled by the learning rate `lr`). Plug into the parabola approximation:

```
change in f  ≈  g·h + ½·c·h²
              =  g·(−lr·g) + ½·c·(lr·g)²
              =  −lr·g²  +  ½·c·lr²·g²
              =  −lr·g² · ( 1 − ½·c·lr )
```

Read it:

- `lr·g²` is always positive (a square), so the first term is **always a decrease**: the slope term wants to take you downhill.
- The curvature term `½·c·lr²·g²` is **positive**: it's the penalty for the curve bending back up while you stride. A *bigger* step makes it grow with `lr²`, faster than the benefit grows with `lr`.
- The whole thing is a decrease only if `(1 − ½·c·lr) > 0`, i.e. **`lr < 2 / c`**.

So there's a speed limit: **learning rate must be below 2 divided by the curvature.** For the bowl `(x − 3)²`, `c = 2`, so the limit is `lr < 1`. Test the claim, with the bowl's slope formula `2(x − 3)` (Lesson 1):

```python
def walk_downhill_on_bowl(learning_rate, number_of_steps=6, starting_position=0.0):
    position = starting_position
    positions_visited = []
    for _ in range(number_of_steps):
        slope = 2 * (position - 3)
        position = position - learning_rate * slope
        positions_visited.append(round(position, 4))
    return positions_visited

for learning_rate in [0.1, 0.5, 0.9, 1.0, 1.1]:
    print(learning_rate, walk_downhill_on_bowl(learning_rate))
```

In [ ]:
# type it here

**Expected output:**
```text
0.1 [0.6, 1.08, 1.464, 1.7712, 2.017, 2.2136]
0.5 [3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
0.9 [5.4, 1.08, 4.536, 1.7712, 3.983, 2.2136]
1.0 [6.0, 0.0, 6.0, 0.0, 6.0, 0.0]
1.1 [6.6, -1.32, 8.184, -3.2208, 10.465, -5.958]
```

Match each row to the maths:

- `lr = 0.1`: creeps toward 3 from one side. Safe but slow.
- `lr = 0.5`: lands exactly on 3 in a single step. (More on why in Step 7.)
- `lr = 0.9`: overshoots to the other side each step, but each overshoot is smaller. Still converging, bouncing.
- `lr = 1.0 = 2/c`: bounces between 0 and 6 forever. Right on the limit.
- `lr = 1.1`: each bounce is *bigger*. **Divergence** (what "my loss exploded" means in practice).

**Connection to Lesson 0:** the update `position − lr·2·(position − 3)` is `nudge(position, 3, alpha = 2·lr)`. The error from the bottom is multiplied by `1 − 2·lr` every step: `0.8, 0, −0.8, −1, −1.2` for `lr = 0.1, 0.5, 0.9, 1.0, 1.1`. Same "multiplier" table as Lesson 0 Step 8 (|multiplier| < 1 converges, negative flips, |multiplier| > 1 explodes). **Gradient descent on a bowl is a linear recurrence**, and now you know where the multiplier comes from: `1 − lr × curvature`.

**Try this (new cell):** change the bowl to `10·(x − 3)²` (slope `20·(x − 3)`, curvature 20). Predict the largest safe learning rate, then test learning rates just below and just above it.

---
## Step 3: Many knobs: a general gradient-descent routine

Cauchy's 1847 version works with a whole *list* of inputs. Take a step against the full gradient vector. Type this reusable routine. It takes the gradient as a *function* (so the same code trains any model):

```python
def gradient_descent(gradient_function, starting_point, learning_rate, number_of_steps):
    point = list(starting_point)
    path = [list(point)]
    for _ in range(number_of_steps):
        slopes = gradient_function(point)
        point = [coordinate - learning_rate * slope for coordinate, slope in zip(point, slopes)]
        path.append(list(point))
    return path

def valley_gradient(point):
    x_value, y_value = point
    return [2 * x_value, 6 * y_value]       # slopes of x**2 + 3*y**2

for learning_rate in [0.1, 0.3, 0.34]:
    path = gradient_descent(valley_gradient, [4, 4], learning_rate, 8)
    print(learning_rate, [[round(value, 3) for value in point] for point in path[:6]])
    print("    after 8 steps:", [round(value, 3) for value in path[-1]])
```

In [ ]:
# type it here

**Expected output:**
```text
0.1 [[4, 4], [3.2, 1.6], [2.56, 0.64], [2.048, 0.256], [1.638, 0.102], [1.311, 0.041]]
    after 8 steps: [0.671, 0.003]
0.3 [[4, 4], [1.6, -3.2], [0.64, 2.56], [0.256, -2.048], [0.102, 1.638], [0.041, -1.311]]
    after 8 steps: [0.003, 0.671]
0.34 [[4, 4], [1.28, -4.16], [0.41, 4.326], [0.131, -4.499], [0.042, 4.679], [0.013, -4.867]]
    after 8 steps: [0.0, 5.474]
```

**Signature:** `gradient_descent` expects (1) a function that takes a point (list) and returns the gradient (list), (2) a starting list, (3) the learning rate, (4) how many steps. It returns the whole `path` so you can inspect it. `zip(point, slopes)` pairs each coordinate with its own slope.

The valley `x² + 3y²` has **different curvature along each axis**: `c = 2` along `x`, `c = 6` along `y`. Each axis behaves like its own 1-D bowl with its own multiplier, `1 − 2·lr` and `1 − 6·lr`:

- `lr = 0.1`: multipliers `0.8` and `0.4`. Both converge. `y` collapses fast, `x` is the slow one.
- `lr = 0.3`: multipliers `0.4` and **`−0.8`**. `y` flips sign each step (visible in the printout) but shrinks. Converging.
- `lr = 0.34`: multipliers `0.32` and **`−1.04`**. `y` bounces with *growing* size. It diverges even though `x` is perfectly happy.

**One learning rate is shared by all the knobs, and the *steepest* direction sets the speed limit** (`lr < 2/6 ≈ 0.333` here). Meanwhile the *shallowest* direction (`x`) is the one that crawls. This tension, steep directions capping the step while shallow ones crawl, is the central difficulty of training real networks, and it's why smarter variants exist (Step 6).

---
## Step 4: Not every landscape is a bowl

Everything so far had one bottom. Take `f(x) = x⁴ − 3x² + x`, a landscape with **two** valleys. Its slope (power rule) is `4x³ − 6x + 1`. Type:

```python
def two_valley_function(x):
    return x ** 4 - 3 * x ** 2 + x

def two_valley_slope(x):
    return 4 * x ** 3 - 6 * x + 1

for starting_position in [-2, 0.5, 2]:
    position = starting_position
    for _ in range(300):
        position = position - 0.01 * two_valley_slope(position)
    print(starting_position, "->", round(position, 4), round(two_valley_function(position), 4))
```

In [ ]:
# type it here

**Expected output:**
```text
-2 -> -1.3008 -3.5139
0.5 -> 1.1309 -1.0702
2 -> 1.1309 -1.0702
```

Starting at `−2` lands in the left valley (`x ≈ −1.30`, height `−3.51`). Starting at `0.5` or `2` lands in the right valley (`x ≈ 1.13`, height `−1.07`), a **local minimum**: lower than its surroundings, but not the lowest point anywhere. Gradient descent only ever knows the slope *right here*, so *where you start decides where you end up*.

- **Linear regression** has a single bowl (squared errors of a line always form one). Descent finds the one answer from anywhere.
- **Neural networks** have enormously complicated landscapes with many dips and flat patches. That's why starting values (initialisation) matter there, and why the lessons on networks keep returning to it.

---
## Step 5: Train a line by walking downhill and check it against Lesson 3's exact answer

Back to the Hooke spring data. Use **mean** squared error (sum divided by the number of points `n`). The chain-rule gradient from Lesson 2 ("2 × error × the input the weight multiplied") averaged over the points:

```
∂MSE/∂w = (2/n) · Σ (prediction − y) · x
∂MSE/∂b = (2/n) · Σ (prediction − y)
```

Build it as a class. Type it all (take your time, it's the growing-file pattern from Lesson 3):

```python
class GradientDescentLine:
    def __init__(self):
        self.slope = 0.0
        self.intercept = 0.0
        self.loss_history = []

    def predict(self, x_value):
        return self.slope * x_value + self.intercept

    def mean_squared_error(self, x_values, y_values):
        total = 0
        for x_value, y_value in zip(x_values, y_values):
            total = total + (self.predict(x_value) - y_value) ** 2
        return total / len(x_values)

    def gradient(self, x_values, y_values):
        slope_total = 0
        intercept_total = 0
        for x_value, y_value in zip(x_values, y_values):
            error = self.predict(x_value) - y_value
            slope_total = slope_total + error * x_value
            intercept_total = intercept_total + error
        number_of_points = len(x_values)
        return [2 * slope_total / number_of_points, 2 * intercept_total / number_of_points]

    def fit(self, x_values, y_values, learning_rate, number_of_steps):
        for step_number in range(number_of_steps):
            loss = self.mean_squared_error(x_values, y_values)
            self.loss_history.append(loss)
            if loss > 1e12:
                print("diverged at step", step_number)
                return
            slope_gradient, intercept_gradient = self.gradient(x_values, y_values)
            self.slope = self.slope - learning_rate * slope_gradient
            self.intercept = self.intercept - learning_rate * intercept_gradient
```

In [ ]:
# type the class here

```python
masses_in_kg = [1, 2, 3, 4, 5]
stretches_in_cm = [2.1, 3.9, 6.2, 7.8, 10.1]

model = GradientDescentLine()
model.fit(masses_in_kg, stretches_in_cm, learning_rate=0.05, number_of_steps=500)

print(model.slope, model.intercept)
print([round(model.loss_history[step], 4) for step in [0, 1, 2, 5, 10, 50, 200, 499]])
```

In [ ]:
# type it here

**Expected output:**
```text
1.9899744054964985 0.05009240427873238
[44.182, 1.5386, 0.1078, 0.0547, 0.0494, 0.0286, 0.0214, 0.0214]
```

Compare with Lesson 3's closed form: slope `1.99`, intercept `0.05`. Gradient descent, which never saw a formula and only ever looked at the local slope, **arrives at the same answer** (to about four decimal places after 500 steps; it keeps getting closer with more). The loss history shows the walk: it plunges from 44 to around 1.5 in one step, then creeps the last bit (the slow shallow direction from Step 3).

**Try this (new cells):** run the same fit with learning rates `0.01`, `0.08`, and `0.09`, using a fresh `GradientDescentLine()` each time and 2000 steps. Print the final slope and intercept. Predict first: one of them should not converge. Then try to say *why that one* in terms of Step 2 (the steepest curvature of this error surface is about 23.7, so the limit is `2 / 23.7 ≈ 0.0845`).

```python
for learning_rate in [0.01, 0.05, 0.08, 0.09]:
    experiment_model = GradientDescentLine()
    experiment_model.fit(masses_in_kg, stretches_in_cm, learning_rate, 2000)
    print(learning_rate, experiment_model.slope, experiment_model.intercept)
```

In [ ]:
# type it here

**Expected output:**
```text
0.01 1.9898525247159426 0.05053243256910116
0.05 1.9899999999999995 0.05000000000000152
0.08 1.9899999999999998 0.05000000000000048
diverged at step 98
0.09 -285297.76669551514 -79023.38355191812
```

`0.01` is still creeping after 2000 steps (slope 1.9899, intercept 0.0505, close but not exact); `0.05` and `0.08` are done; **`0.09` diverged** (the message prints and the numbers are huge), as Step 2 predicted for anything above ~0.0845. The speed limit isn't a rule of thumb, it falls out of the algebra.

---
## Step 6: Guided exploration: the narrow-valley problem and momentum

Make the valley much more stretched: `f(x, y) = x² + 100·y²`. Curvature is `2` along `x` and `200` along `y`, so the speed limit is `2/200 = 0.01`, but `x`, using that tiny step, is hardly moving (multiplier `1 − 2·0.0099 ≈ 0.98`). Count how many steps plain descent needs to get the height below 0.000001 from `[5, 1]`:

```python
def narrow_valley_height(point):
    return point[0] ** 2 + 100 * point[1] ** 2

def narrow_valley_gradient(point):
    return [2 * point[0], 200 * point[1]]

def steps_needed_plain(learning_rate, maximum_steps=20000):
    point = [5.0, 1.0]
    for step_number in range(maximum_steps):
        if narrow_valley_height(point) < 0.000001:
            return step_number
        slopes = narrow_valley_gradient(point)
        point = [coordinate - learning_rate * slope for coordinate, slope in zip(point, slopes)]
    return None

print(steps_needed_plain(0.0099))
print(steps_needed_plain(0.005))
```

In [ ]:
# type it here

**Expected output:**
```text
463
848
```

About **463 steps** at the best safe rate, and slower (848) for the smaller one. The slow direction sets the pace.

**A named alternative: momentum** (Boris Polyak, 1964, the "heavy ball" method). Instead of stepping purely by the current slope, keep a **velocity** that remembers recent steps: pushes in a consistent direction build up speed, while the back-and-forth bouncing across the steep direction cancels out.

```
velocity = beta · velocity − learning_rate · gradient
position = position + velocity
```

`beta` (between 0 and 1) is how much of the old velocity survives; 0 gives back plain descent. Type it and compare:

```python
def steps_needed_with_momentum(learning_rate, beta, maximum_steps=20000):
    point = [5.0, 1.0]
    velocity = [0.0, 0.0]
    for step_number in range(maximum_steps):
        if narrow_valley_height(point) < 0.000001:
            return step_number
        slopes = narrow_valley_gradient(point)
        velocity = [beta * old_velocity - learning_rate * slope for old_velocity, slope in zip(velocity, slopes)]
        point = [coordinate + step for coordinate, step in zip(point, velocity)]
    return None

print(steps_needed_with_momentum(0.004, 0.85))
print(steps_needed_with_momentum(0.004, 0.0))
```

In [ ]:
# type it here

**Expected output:**
```text
112
1061
```

Momentum with `lr = 0.004` and `beta = 0.85` needs about **112 steps**, roughly 4× fewer than the best plain run (463). The second line sets `beta = 0`, which reduces to plain descent at the same learning rate, and needs about **1061 steps**: same step size, ten times slower without the velocity. *Return to the original plain version for the next step.* Modern optimisers (RMSprop, **Adam**, 2014) are cleverer variations on this theme: remember past gradients, adapt the step per knob.

**Try this (new cells):** vary `beta` over `0.5, 0.8, 0.9, 0.95, 0.99` at `lr = 0.004`. Does bigger always help? (Hint: with too much memory the ball overshoots and oscillates, a Lesson 0 multiplier going the wrong way.)

---
## Step 7: Newton's method: use the curvature and sometimes jump straight to the bottom

If you know the curvature `c`, the parabola approximation has its bottom at `x − g/c`. That's a step with `lr = 1/c`. For a true bowl it lands **exactly on the bottom in one step**: the multiplier `1 − c·lr` becomes `0`. This is Newton's method (1680s), turned from root-finding into minimising:

```python
def newton_step(position, slope, curvature):
    return position - slope / curvature

starting_position = 0.0
slope_there = 2 * (starting_position - 3)
print(newton_step(starting_position, slope_there, curvature=2))

# in two dimensions: use each axis's own curvature (2 and 6) for the valley x**2 + 3*y**2
starting_point = [4, 4]
slopes_there = valley_gradient(starting_point)
curvatures = [2, 6]
print([coordinate - slope / curvature for coordinate, slope, curvature in zip(starting_point, slopes_there, curvatures)])
```

In [ ]:
# type it here

**Expected output:**
```text
3.0
[0.0, 0.0]
```

Both land on the exact bottom (`3` and `[0, 0]`) in one step. In Lesson 0's language, Newton's method is a **nudge with `alpha = 1`**: close the *whole* gap. Why doesn't everyone use it? A model with a million knobs has a million-by-million table of curvatures (every knob against every other knob), far too big to compute or store. So large-scale training uses cheap approximations: momentum, Adam, and similar.

---
## Step 8: Challenge

Take `f(x, y) = 2x² + 5y²`.

1. Slopes (by hand): `∂f/∂x = ?` and `∂f/∂y = ?`. Curvatures: `?` and `?`.
2. Using Step 2's rule `lr < 2/c`, predict the **largest safe learning rate** for gradient descent on this landscape.
3. Test your prediction: run `gradient_descent` from `[3, 3]` for 40 steps with a learning rate just below it, and just above it. (Compare the final heights.)
4. Using the multipliers `1 − c·lr` for each axis, find the learning rate that makes the two multipliers equal in size (`|1 − 4·lr| = |1 − 10·lr|`, taking opposite signs). Why might that be the best single learning rate? Test it.

```python
# your attempt here
```

In [ ]:
# type it here

---
## Where this goes next

You can now explain training mechanically: slope tells you the direction, curvature caps the step size, and the multiplier `1 − lr × curvature` governs the whole journey. Lesson 5 changes gears to **linear algebra**, the maths of handling many numbers at once, so that "a list of knobs" and "a list of data features" become single objects (vectors and matrices) and the many-knob versions of Lessons 3 and 4 become one-line equations.

Once you're comfortable, one thing to probe: this lesson used the *whole* dataset to compute each gradient. Search "stochastic gradient descent" and "mini-batch" to find out what happens when you estimate the gradient from just a few data points per step (it's noisier but much cheaper, and it's how large models are trained).

---
# Solutions to the Step 8 challenge (try it first)

1. `∂f/∂x = 4x`, `∂f/∂y = 10y`. Curvatures: `c = 4` (x) and `c = 10` (y).
2. Limit set by the steepest direction: `lr < 2/10 = 0.2`.
3. Test:

```python
def challenge_gradient(point):
    return [4 * point[0], 10 * point[1]]

def challenge_height(point):
    return 2 * point[0] ** 2 + 5 * point[1] ** 2

for learning_rate in [0.19, 0.21, 1 / 7]:
    final_point = gradient_descent(challenge_gradient, [3, 3], learning_rate, 40)[-1]
    print(round(learning_rate, 4), final_point, challenge_height(final_point))
```

**Output:**
```text
0.19 [4.84809278746971e-25, 0.044342648824303804] 0.009831352523777658
0.21 [4.3845049119927543e-32, 135.77776670452747] 92178.00965634544
0.1429 [5.728617731847284e-15, 5.728617731847234e-15] 2.2971942782344298e-28
```

`0.19` converges slowly (the `y` multiplier is `1 − 1.9 = −0.9`), `0.21` diverges (multiplier `−1.1`, and the `y` coordinate has grown to about 136). **4.** Setting `1 − 4·lr = −(1 − 10·lr)` gives `lr = 2/14 = 1/7 ≈ 0.143`, where the multipliers are `+3/7` and `−3/7`: **both axes shrink at the same rate**, the best the single shared learning rate can do. After 40 steps the height is about `2e-28`, essentially exactly zero. The steeper the ratio between the two curvatures, the worse the best achievable rate gets (Step 6's `2 : 200` valley), which is the problem momentum and Adam attack.